# SEC-BERT report-grouped 5-fold OOF
Attach the private dataset containing `sec_bert_oof_kaggle_bundle.zip`, enable GPU and Internet, then run all. Each completed fold is saved; rerunning resumes unfinished folds.

In [ ]:
from pathlib import Path
import shutil, zipfile
input_root = Path('/kaggle/input')
project = Path('/kaggle/working/selective-audit-routing')
bundles = list(input_root.rglob('sec_bert_oof_kaggle_bundle.zip'))
configs = list(input_root.rglob('encoder_sec_domain_oof_v1.json'))
if (project/'configs/encoder_sec_domain_oof_v1.json').exists():
    print('Reusing working project so completed folds are preserved')
elif len(bundles) == 1:
    project.mkdir(parents=True)
    with zipfile.ZipFile(bundles[0]) as archive: archive.extractall(project)
elif len(configs) == 1:
    shutil.copytree(configs[0].parent.parent, project)
else:
    raise FileNotFoundError(f'Attach the OOF dataset. bundles={bundles}, configs={configs}')
required = [project/'configs/encoder_sec_domain_oof_v1.json', project/'src/train_encoder_sec_domain_oof_v1.py', project/'data/processed/pcaob_development_oof_v1.csv', project/'data/interim/pcaob_rag_pilot_v1_audit.csv']
missing = [str(path) for path in required if not path.exists()]
assert not missing, missing
print('Project ready:', project)

In [ ]:
%pip install -q --upgrade 'transformers>=4.45,<6' sentencepiece scikit-learn pandas
import torch
assert torch.cuda.is_available(), 'Enable GPU in Kaggle Settings'
print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
import subprocess, sys
command = [sys.executable, str(project/'src/train_encoder_sec_domain_oof_v1.py'), '--cache-dir', '/kaggle/working/hf_cache']
subprocess.run(command, cwd=project, check=True)

In [ ]:
archive = shutil.make_archive('/kaggle/working/sec_bert_oof_results', 'zip', project/'results')
print('Download:', archive)